In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import pipeline

: 

In [5]:
if torch.cuda.is_available():
    print(f"GPU is available! Using: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

GPU is available! Using: NVIDIA GeForce RTX 2060


In [6]:
checkpoint_path = 'dataset/filteredData/'
structured_file = os.path.join(checkpoint_path, 'ami_cohort_structured_features.csv')
notes_file = os.path.join(checkpoint_path, 'ami_cohort_discharge_notes.csv')

In [9]:
notes_df = pd.read_csv(notes_file)
notes_df.head()

,hadm_id,text
0,27897940,\nName: ___ Unit No: ___\n \...
1,26913865,\nName: ___ Unit No: ___\n \nAdmi...
2,24947999,\nName: ___ Unit No: ___\n \nAdmi...
3,25242409,\nName: ___ Unit No: ___\n \nAdmi...
4,25911675,\nName: ___ Unit No: ___\n \nAdmi...


# Sentiment Analysis Pipeline

load model

In [11]:
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

d:\IIT\IRP\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\damit\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular 

text data

In [12]:
notes_df['text'] = notes_df['text'].astype(str).fillna('')
text_list = notes_df['text'].tolist()

In [13]:
len(text_list)

27677

In [14]:
sentiment_results = sentiment_pipe(text_list, batch_size=32, truncation=True)

convert resultss to score

In [15]:
def process_sentiment(result):
    score = result['score']
    if result['label'] == 'NEGATIVE':
        return -score
    return score

In [16]:
notes_df['sentiment_score'] = [process_sentiment(r) for r in sentiment_results]

In [17]:
sentiment_notes_file = os.path.join(checkpoint_path, 'ami_cohort_notes_with_sentiment.csv')
notes_df.to_csv(sentiment_notes_file, index=False)
notes_df.head()

,hadm_id,text,sentiment_score
0,27897940,\nName: ___ Unit No: ___\n \...,-0.996072
1,26913865,\nName: ___ Unit No: ___\n \nAdmi...,-0.996290
2,24947999,\nName: ___ Unit No: ___\n \nAdmi...,-0.990710
3,25242409,\nName: ___ Unit No: ___\n \nAdmi...,-0.984590
4,25911675,\nName: ___ Unit No: ___\n \nAdmi...,-0.993728
